You can download and run this notebook locally, or you can run it for free in a cloud environment using Colab or Sagemaker Studio Lab:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kirbyju/TCIA_Notebooks/blob/main/TCIA_REST_API_Queries.ipynb)

[![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github.com/kirbyju/TCIA_Notebooks/blob/main/TCIA_REST_API_Queries.ipynb)

# Summary

Access to large, high-quality datasets is essential for researchers to understand disease and precision medicine pathways, especially in cancer. However, HIPAA constraints make sharing medical images outside an individual institution complex. [The Cancer Imaging Archive (TCIA)](https://www.cancerimagingarchive.net/) is a public service funded by the National Cancer Institute that addresses this challenge by providing hosting and de-identification services that take major burdens of data sharing off researchers.

Datasets that are curated and published by TCIA data are hosted in a variety of different databases depending on data type and licensing.  **This notebook is focused on accessing TCIA's public DICOM data from the NCI Imaging Data Commons (IDC).**  If you're interested in additional TCIA notebooks and coding examples, check out the tutorials at https://github.com/kirbyju/TCIA_Notebooks.

# 1 Learn about Available Collections on the TCIA Website

[Browsing Collections](https://www.cancerimagingarchive.net/collections) and viewing [Analysis Results](https://www.cancerimagingarchive.net/tcia-analysis-results/) of TCIA datasets can be a useful prerequisite to the steps in this notebook. These pages will help you quickly identify datasets that you'd like to explore further using the steps below.  They can also help you find valuable supporting data (e.g. clinical spreadsheets and non-DICOM segmentation data) and answer the most common questions you might have about the datasets.  There is also a [REST API](https://www.cancerimagingarchive.net/collection-manager-rest-api/) to work with these types metadata, but that is not the focus of this notebook.

# 2 Accessing public DICOM data via NCI's Imaging Data Commons
TCIA has partnered with the [NCI Imaging Data Commons (IDC)](https://portal.imaging.datacommons.cancer.gov/) to host and distribute our public DICOM data.  You can learn more about IDC's capabilities to explore, subset, analyze and share data at https://learn.canceridc.dev/getting-started-with-idc.  

If you're interested in LLM "Skills" you should also check out the [IDC Claude Skill](https://github.com/ImagingDataCommons/idc-claude-skill/blob/main/USAGE.md).

In this notebook we will stick to the basics for querying and downloading using the [**idc-index**](https://github.com/ImagingDataCommons/idc-index) python package.  You can find more information about doing complex searches on any DICOM element using Google BigQuery in [this notebook developed by the IDC team](https://github.com/ImagingDataCommons/IDC-Tutorials/blob/master/notebooks/getting_started/part3_exploring_cohorts.ipynb).  There is also a [web-based IDC GUI](https://portal.imaging.datacommons.cancer.gov/explore/).

**Important note:** In addition to supporting TCIA's public DICOM data, IDC also hosts data from other sources, and some derived analyses of TCIA Collections that were not curated and published by TCIA.  You can see a full list of all of their datasets at https://portal.imaging.datacommons.cancer.gov/collections/.  Datasets originally published by IDC leverage Zenodo for DOIs and summary landing pages.  You can view the full list of these datasets in the [IDC Zenodo Community](https://zenodo.org/communities/nci-idc/records).

# 3 Setup

The following cells install and import [**idc-index**](https://github.com/ImagingDataCommons/idc-index) which contain a variety of useful functions for accessing TCIA datasets stored in IDC via Jupyter/Python.

In [1]:
import sys

# install tcia utils
!{sys.executable} -m pip install --upgrade -q idc-index

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 87.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
from idc_index import IDCClient

idc_client = IDCClient()

# 4 Basic Query Examples
By default, idc-index functions return results as dataframes.  Let's take a quick look at what columns are available.

In [3]:
print(idc_client.index.columns.values)

['collection_id' 'analysis_result_id' 'PatientID' 'SeriesInstanceUID'
 'StudyInstanceUID' 'source_DOI' 'PatientAge' 'PatientSex' 'StudyDate'
 'StudyDescription' 'BodyPartExamined' 'Modality' 'Manufacturer'
 'ManufacturerModelName' 'SeriesDate' 'SeriesDescription' 'SeriesNumber'
 'instanceCount' 'license_short_name' 'aws_bucket' 'crdc_series_uuid'
 'series_aws_url' 'series_size_MB']


## 4.1 Get a list of available datasets


In [4]:
collections = idc_client.index['collection_id'].unique()
analysis_results = idc_client.index['analysis_result_id'].unique()

print("Available Collections:")
display(collections)

print("\nAvailable Analysis Results:")
display(analysis_results)

Available Collections:


array(['cptac_pda', 'pancreas_ct', 'qiba_ct_liver_phantom', 'tcga_kirc',
       'tcga_stad', 'tcga_brca', 'ct4harmonization_multicentric',
       'ct_colonography', 'cc_radiomics_phantom_3', 'cmb_crc',
       'varepop_apollo', 'tcga_blca', 'rider_lung_ct',
       'nsclc_radiogenomics', 'tcga_ucec', 'phantom_fda', 'upenn_gbm',
       'acrin_nsclc_fdg_pet', 'cc_tumor_heterogeneity', 'naf_prostate',
       'hcc_tace_seg', 'vestibular_schwannoma_seg', 'cptac_ucec',
       'rider_pilot', 'pdmr_texture_analysis', 'midrc_ricord_1a',
       'vestibular_schwannoma_mc_rc', 'cmb_lca', 'acrin_flt_breast',
       'ct_vs_pet_ventilation_imaging', 'icdc_glioma', 'anti_pd_1_lung',
       'lung_pet_ct_dx', 'prostate_mri_us_biopsy', 'tcga_lihc',
       'nsclc_radiomics', 'tcga_ov', 'qin_breast_dce_mri',
       'cc_radiomics_phantom_2', 'stageii_colorectal_ct', 'cmb_gec',
       'cptac_aml', 'tcga_kirp', 'cmb_brca', 'pdmr_833975_119_r',
       'breast_mri_nact_pilot', 'qiba_ct_1c', 'remind', 'c4kc_kits',


Available Analysis Results:


array([None, 'Prostate-MRI-US-Biopsy-DICOM-Annotations', 'NLSTSeg',
       'Lung-PET-CT-Dx-Annotations', 'Pancreas-CT-SEG', 'TCGA-GBM360',
       'ProstateX-Targets', 'CPTAC-UCEC-Tumor-Annotations',
       'QIBA-VolCT-1B', 'CPTAC-CCRCC-Tumor-Annotations',
       'DICOM-SR-Breast-Clinical', 'CPTAC-PDA-Tumor-Annotations',
       'QIN-LungCT-Seg', 'RMS-Mutation-Prediction-Expert-Annotations',
       'NLST-Sybil', 'RIDER-LungCT-Seg', 'PROSTATEx-Seg-Zones',
       'PROSTATEx-Seg-HiRes', 'Pan-Cancer-Nuclei-Seg-DICOM',
       'TCGA-SBU-TIL-Maps', 'nnU-Net-BPR-annotations',
       'BAMF-AIMI-Annotations', 'TotalSegmentator-CT-Segmentations',
       'DICOM-LIDC-IDRI-Nodules'], dtype=object)

Let's make this list a little more useful by summarizing other key information contained in idc-index.  First, we'll group the relevant data from `idc_client.index` by `collection_id`. For each collection, find unique `source_DOI` and `license_short_name` values (listed as arrays), count unique `PatientID`, `StudyInstanceUID`, and `SeriesInstanceUID`, and sum `series_size_MB`. We'll also add a new column named 'Type' with the value 'collection'.

In [5]:
collections_df = idc_client.index.groupby('collection_id').agg(
    Unique_Source_DOIs=('source_DOI', lambda x: x.dropna().unique().tolist()),
    Unique_Licenses=('license_short_name', lambda x: x.dropna().unique().tolist()),
    Unique_Body_Parts=('BodyPartExamined', lambda x: x.dropna().unique().tolist()),
    Unique_Modalities=('Modality', lambda x: x.dropna().unique().tolist()),
    Num_Patients=('PatientID', 'nunique'),
    Num_Studies=('StudyInstanceUID', 'nunique'),
    Num_Series=('SeriesInstanceUID', 'nunique'),
    Total_Series_Size_MB=('series_size_MB', 'sum')
).reset_index()

collections_df['Type'] = 'collection'

# sort smallest to largest
collections_df = collections_df.sort_values(by='Total_Series_Size_MB', ascending=True)

print("Aggregated Collection Details:")
display(collections_df.head())

Aggregated Collection Details:


,collection_id,Unique_Source_DOIs,Unique_Licenses,Unique_Body_Parts,Unique_Modalities,Num_Patients,Num_Studies,Num_Series,Total_Series_Size_MB,Type
110,qin_pet_phantom,[10.7937/k9/tcia.2015.zpukhckb],[CC BY 3.0],[PHANTOM],[PT],2,4,44,246.28,collection
96,prostate_3t,[10.7937/k9/tcia.2015.qjtv5il5],[CC BY 3.0],[PROSTATE],[MR],64,64,64,297.73,collection
113,rider_breast_mri,[10.7937/k9/tcia.2015.h1sxnuxl],[CC BY 3.0],[BREAST],[MR],5,10,40,401.60,collection
117,rider_phantom_pet_ct,[10.7937/k9/tcia.2015.8wg2kn4w],[CC BY 3.0],[PHANTOM],"[CT, PT]",20,20,60,722.94,collection
69,lung_phantom,"[10.7937/k9/tcia.2015.1buvfjr7, 10.7937/k9/tci...",[CC BY 3.0],[],"[SEG, CT]",1,1,109,982.86,collection


Now we'll summarize Analysis Results in the same manner using `analysis_result_id`.

In [6]:
analysis_results_df = idc_client.index.groupby('analysis_result_id').agg(
    Unique_Source_DOIs=('source_DOI', lambda x: x.dropna().unique().tolist()),
    Unique_Licenses=('license_short_name', lambda x: x.dropna().unique().tolist()),
    Unique_Body_Parts=('BodyPartExamined', lambda x: x.dropna().unique().tolist()),
    Unique_Modalities=('Modality', lambda x: x.dropna().unique().tolist()),
    Num_Patients=('PatientID', 'nunique'),
    Num_Studies=('StudyInstanceUID', 'nunique'),
    Num_Series=('SeriesInstanceUID', 'nunique'),
    Total_Series_Size_MB=('series_size_MB', 'sum')
).reset_index()

analysis_results_df['Type'] = 'analysis_result'

print("Aggregated Analysis Results Details:")
display(analysis_results_df.head())

Aggregated Analysis Results Details:


,analysis_result_id,Unique_Source_DOIs,Unique_Licenses,Unique_Body_Parts,Unique_Modalities,Num_Patients,Num_Studies,Num_Series,Total_Series_Size_MB,Type
0,BAMF-AIMI-Annotations,[10.5281/zenodo.8345959],[CC BY 4.0],"[ABDOMEN, BRAIN, LIVER, CHEST, LUNG, HEAD, PEL...",[SEG],4226,6972,8202,26878.83,analysis_result
1,CPTAC-CCRCC-Tumor-Annotations,[10.7937/skq4-qx48],[CC BY 4.0],"[ABDOMEN, CHEST, HEADNECK, PELVIS, EXTREMITY]",[RTSTRUCT],60,73,636,27.36,analysis_result
2,CPTAC-PDA-Tumor-Annotations,[10.7937/bw9v-bx61],[CC BY 4.0],"[ABDOMEN, PELVIS, CHEST, NECKCHESTABDPELV]",[RTSTRUCT],103,120,536,13.56,analysis_result
3,CPTAC-UCEC-Tumor-Annotations,[10.7937/89m3-kq43],[CC BY 4.0],"[PELVIS, ABDOMEN, CHEST, NECK, NECKCHESTABDOME...",[RTSTRUCT],72,100,617,33.64,analysis_result
4,DICOM-LIDC-IDRI-Nodules,[10.7937/tcia.2018.h7umfurq],[CC BY 3.0],[CHEST],"[SEG, SR]",875,883,13718,2510.31,analysis_result


And now we can combine `collections_df` and `analysis_results_df` in a single dataframe.


In [7]:
collections_df = collections_df.rename(columns={'collection_id': 'ID'})
analysis_results_df = analysis_results_df.rename(columns={'analysis_result_id': 'ID'})

combined_summary_df = pd.concat([collections_df, analysis_results_df], ignore_index=True)

print("Combined Summary Table (first 5 rows):")
display(combined_summary_df.head())

Combined Summary Table (first 5 rows):


,ID,Unique_Source_DOIs,Unique_Licenses,Unique_Body_Parts,Unique_Modalities,Num_Patients,Num_Studies,Num_Series,Total_Series_Size_MB,Type
0,qin_pet_phantom,[10.7937/k9/tcia.2015.zpukhckb],[CC BY 3.0],[PHANTOM],[PT],2,4,44,246.28,collection
1,prostate_3t,[10.7937/k9/tcia.2015.qjtv5il5],[CC BY 3.0],[PROSTATE],[MR],64,64,64,297.73,collection
2,rider_breast_mri,[10.7937/k9/tcia.2015.h1sxnuxl],[CC BY 3.0],[BREAST],[MR],5,10,40,401.60,collection
3,rider_phantom_pet_ct,[10.7937/k9/tcia.2015.8wg2kn4w],[CC BY 3.0],[PHANTOM],"[CT, PT]",20,20,60,722.94,collection
4,lung_phantom,"[10.7937/k9/tcia.2015.1buvfjr7, 10.7937/k9/tci...",[CC BY 3.0],[],"[SEG, CT]",1,1,109,982.86,collection


## 4.2 Get a list of available body parts examined
Next let's see what kinds of values we have in BodyPartExamined.  Unfortunately, there is a rather significant volume of data submitted to us where this data is not populated.

In [8]:
body_part_counts = idc_client.index['BodyPartExamined'].value_counts(dropna=False)

print("DICOM series counts of each unique BodyPartExamined (including NaN):")
display(body_part_counts)

DICOM series counts of each unique BodyPartExamined (including NaN):


,count
BodyPartExamined,
None,453957
CHEST,356133
BREAST,106757
PROSTATE,23008
LUNG,10971
...,...
CTSPINE,1
FEMUR,1
KNEE,1


## 4.3 Get a list of available modalities
And now let's see what Modality values we have.  These abbreviations are explained in more detail in [Part 16 Chapter D of the DICOM standard](https://dicom.nema.org/medical/dicom/current/output/chtml/part16/chapter_d.html).

In [9]:
modality_counts = idc_client.index['Modality'].value_counts(dropna=False)

print("DICOM series counts of each unique Modality:")
display(modality_counts)

DICOM series counts of each unique Modality:


,count
Modality,
SR,270687
CT,252008
SEG,188013
MR,124072
SM,71132
MG,48125
CR,12416
ANN,7108
RTSTRUCT,4938


## 4.4 Summarize patient age and sex characteristics
It's **important to note** that often times this information is missing or may occasionally be filled with non-sense values by the folks who acquired the images.  If you can find an accompanying spreadsheet or other source of patient information that should pretty much always be assumed more accurate than this information coming from the DICOM headers, but sometimes it's the only thing available.

In [10]:
patient_sex_counts = idc_client.index['PatientSex'].value_counts(dropna=False)

print("DICOM series counts of each unique PatientSex:")
display(patient_sex_counts)

DICOM series counts of each unique PatientSex:


,count
PatientSex,
None,693136
F,175709
M,106506
O,18552
0000,85
U,77
6657,8


Let's look at patient age now.  Note that they may be encoded differently with some represented as days (e.g. 399D), months (e.g. 272M) or years (066Y).  Again, many of these values are not populated.

In [11]:
# Get PatientAge data
patient_age = idc_client.index['PatientAge']

# group by age
patient_age = patient_age.value_counts(dropna=False)

display(patient_age)

,count
PatientAge,
None,799606
066Y,13570
056Y,12311
000Y,10005
046Y,6842
...,...
095D,1
365D,1
463D,1


## 4.5 Review metadata for specific DICOM Series

Each row returned by idc-index represents a specific DICOM Series.  Let's say that you're interested in looking at MR images of the breast.  You might start with something like this:

In [12]:
# filter idc_client.index for Modality = MR and BodyPartExamined = BREAST
df = idc_client.index[(idc_client.index['Modality'] == 'MR')
 & (idc_client.index['BodyPartExamined'] == 'BREAST')]

display(df)

,collection_id,analysis_result_id,PatientID,SeriesInstanceUID,StudyInstanceUID,source_DOI,PatientAge,PatientSex,StudyDate,StudyDescription,...,ManufacturerModelName,SeriesDate,SeriesDescription,SeriesNumber,instanceCount,license_short_name,aws_bucket,crdc_series_uuid,series_aws_url,series_size_MB
5,tcga_brca,None,TCGA-AR-A1AQ,1.3.6.1.4.1.14519.5.2.1.3344.4002.222132418953...,1.3.6.1.4.1.14519.5.2.1.3344.4002.100294194044...,10.7937/k9/tcia.2016.ab2nazrp,050Y,F,2003-05-07,"MRI BREAST, BILATERAL",...,SIGNA HDx,2003-05-07,(2126/8/113)-(2126/8/1),100,112,CC BY 3.0,idc-open-data,ffab3f5a-5b44-4823-a805-62df2e4b578d,s3://idc-open-data/ffab3f5a-5b44-4823-a805-62d...,15.09
61,qin_breast_dce_mri,None,QIN-Breast-DCE-MRI-BC14,1.3.6.1.4.1.14519.5.2.1.2103.7010.108224585197...,1.3.6.1.4.1.14519.5.2.1.2103.7010.121064295202...,10.7937/k9/tcia.2014.a2n1ixox,041Y,F,1997-11-10,Breast CE,...,TrioTim,1997-11-10,twist_20s_dyn_TRA (h20 ex B17)_TT=335.5s,39,120,CC BY 3.0,idc-open-data,6e9d17ac-1a7a-423f-8e99-e43cc61855d1,s3://idc-open-data/6e9d17ac-1a7a-423f-8e99-e43...,24.93
118,breast_diagnosis,None,BreastDx-01-0044,1.3.6.1.4.1.14519.5.2.1.4792.2001.251897966735...,1.3.6.1.4.1.14519.5.2.1.4792.2001.171256348471...,10.7937/k9/tcia.2015.sdnrqxxr,None,F,2008-05-27,MRI Right Breast with and without Contrast,...,Achieva,2008-05-27,T2W_TSE SENSE,401,88,CC BY 3.0,idc-open-data,6cbb1693-0e81-415c-b10b-173415ba9ac2,s3://idc-open-data/6cbb1693-0e81-415c-b10b-173...,46.40
126,breast_mri_nact_pilot,None,UCSF-BR-06,1.3.6.1.4.1.14519.5.2.1.7695.2311.338942706262...,1.3.6.1.4.1.14519.5.2.1.7695.2311.332840451761...,10.7937/k9/tcia.2016.qhsyhjky,071Y,F,1991-11-26,"MR BREAS, UNIT",...,GENESIS_SIGNA,1991-11-26,Dynamic-3dfgre: SER,41000,60,CC BY 3.0,idc-open-data,7edee498-0dc3-46a1-bbf5-57318f3e73a7,s3://idc-open-data/7edee498-0dc3-46a1-bbf5-573...,8.24
128,tcga_brca,None,TCGA-BH-A0E0,1.3.6.1.4.1.14519.5.2.1.6450.4002.301512648183...,1.3.6.1.4.1.14519.5.2.1.6450.4002.783686167303...,10.7937/k9/tcia.2016.ab2nazrp,038Y,F,2003-04-06,BREAST MRI,...,SIGNA EXCITE,2003-04-06,3 plane loc,1,29,CC BY 3.0,idc-open-data,50026c25-4a00-4d4c-937b-6cb54bd69564,s3://idc-open-data/50026c25-4a00-4d4c-937b-6cb...,3.96
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994046,acrin_6698,None,ACRIN-6698-220471,1.3.6.1.4.1.14519.5.2.1.7695.4164.889586546012...,1.3.6.1.4.1.14519.5.2.1.7695.4164.306084077852...,10.7937/tcia.kk02-6d95,None,F,2001-11-22,ACRIN-6698_ISPY2_MRI_T3,...,Avanto,2001-11-22,ACRIN-6698: AX DWI 6698,2,140,CC BY 4.0,idc-open-data,da613be4-1153-4bec-aa83-8e045efb74c8,s3://idc-open-data/da613be4-1153-4bec-aa83-8e0...,29.22
994049,acrin_6698,None,ACRIN-6698-689480,1.3.6.1.4.1.14519.5.2.1.7695.4164.102728851057...,1.3.6.1.4.1.14519.5.2.1.7695.4164.105433616534...,10.7937/tcia.kk02-6d95,None,F,2002-05-15,ACRIN-6698_ISPY2_MRI_T1,...,Achieva,2002-05-15,ISPY2: VOLSER: uni-lateral cropped: SER,161000,158,CC BY 4.0,idc-open-data,23104e1f-7290-4ae8-b2dc-06d75b990bd9,s3://idc-open-data/23104e1f-7290-4ae8-b2dc-06d...,22.58
994051,acrin_6698,None,ACRIN-6698-710197,1.3.6.1.4.1.14519.5.2.1.7695.4164.278943728845...,1.3.6.1.4.1.14519.5.2.1.7695.4164.653729536721...,10.7937/tcia.kk02-6d95,None,F,2002-04-06,ACRIN-6698_ISPY2_MRI_T0,...,Achieva,2002-04-06,ACRIN-6698: DWI_SSh 6698 90,701,200,CC BY 4.0,idc-open-data,93584fe6-18c1-408f-baf2-77a1921f0aab,s3://idc-open-data/93584fe6-18c1-408f-baf2-77a...,27.94
994061,acrin_6698,None,ACRIN-6698-786753,1.3.6.1.4.1.14519.5.2.1.7695.4164.897879436349...,1.3.6.1.4.1.14519.5.2.1.7695.4164.773267338236...,10.7937/tcia.kk02-6d95,None,F,2001-07-07,ACRIN-6698_ISPY2_MRI_T1,...,Achieva,2001-07-07,ACRIN-6698: DWI_SSh,801,136,CC BY 4.0,idc-open-data,c8bf0209-730b-4145-9a87-548fb929b8b6,s3://idc-open-data/c8bf0209-730b-4145-9a87-548...,23.74


You might then further refine your dataset by focusing only on T2 acquisitions.


In [13]:
# filter df to only include rows with SeriesDescription or StudyDescription that contain "T2" (ignore case)
breast_mr_t2_df = df[df['SeriesDescription'].str.contains('T2', case=False, na=False) |
        df['StudyDescription'].str.contains('T2', case=False, na=False)]

display(breast_mr_t2_df)

,collection_id,analysis_result_id,PatientID,SeriesInstanceUID,StudyInstanceUID,source_DOI,PatientAge,PatientSex,StudyDate,StudyDescription,...,ManufacturerModelName,SeriesDate,SeriesDescription,SeriesNumber,instanceCount,license_short_name,aws_bucket,crdc_series_uuid,series_aws_url,series_size_MB
118,breast_diagnosis,None,BreastDx-01-0044,1.3.6.1.4.1.14519.5.2.1.4792.2001.251897966735...,1.3.6.1.4.1.14519.5.2.1.4792.2001.171256348471...,10.7937/k9/tcia.2015.sdnrqxxr,None,F,2008-05-27,MRI Right Breast with and without Contrast,...,Achieva,2008-05-27,T2W_TSE SENSE,401,88,CC BY 3.0,idc-open-data,6cbb1693-0e81-415c-b10b-173415ba9ac2,s3://idc-open-data/6cbb1693-0e81-415c-b10b-173...,46.40
590,breast_mri_nact_pilot,None,UCSF-BR-28,1.3.6.1.4.1.14519.5.2.1.7695.2311.324054068278...,1.3.6.1.4.1.14519.5.2.1.7695.2311.872371497493...,10.7937/k9/tcia.2016.qhsyhjky,060Y,F,1993-06-22,"MR BREAS, UNIT",...,GENESIS_SIGNA,1993-06-22,T2-FSE-Sagittal,3,36,CC BY 3.0,idc-open-data,a7dbc4a7-8d1f-4d0e-9b0a-7883c6095330,s3://idc-open-data/a7dbc4a7-8d1f-4d0e-9b0a-788...,4.88
996,tcga_brca,None,TCGA-AO-A12F,1.3.6.1.4.1.14519.5.2.1.9203.4002.256306564845...,1.3.6.1.4.1.14519.5.2.1.9203.4002.240753407080...,10.7937/k9/tcia.2016.ab2nazrp,036Y,F,2000-10-16,None,...,SIGNA EXCITE,2000-10-16,T2 left breast,2,38,CC BY 3.0,idc-open-data,756f8af4-df01-45f1-ac20-ee48a953ef0f,s3://idc-open-data/756f8af4-df01-45f1-ac20-ee4...,5.15
1245,breast_mri_nact_pilot,None,UCSF-BR-57,1.3.6.1.4.1.14519.5.2.1.7695.2311.122433364947...,1.3.6.1.4.1.14519.5.2.1.7695.2311.378799439450...,10.7937/k9/tcia.2016.qhsyhjky,040Y,F,1994-01-12,MR BREASTUNI UE,...,GENESIS_SIGNA,1994-01-12,T2-FSE-Sagittal,2,36,CC BY 3.0,idc-open-data,5ea45ff8-e675-4771-a654-ad08fe10e897,s3://idc-open-data/5ea45ff8-e675-4771-a654-ad0...,4.88
1514,tcga_brca,None,TCGA-E2-A15J,1.3.6.1.4.1.14519.5.2.1.3023.4002.327727025527...,1.3.6.1.4.1.14519.5.2.1.3023.4002.191553757781...,10.7937/k9/tcia.2016.ab2nazrp,None,None,2002-06-20,"MR BREAST, BILATERAL W/WO CONT",...,SIGNA EXCITE,2002-06-20,SAG T2 (FAT-SAT) RIGHT,4,46,CC BY 3.0,idc-open-data,fa47cb46-6e1c-455e-9cf3-e975c7ad4373,s3://idc-open-data/fa47cb46-6e1c-455e-9cf3-e97...,6.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994025,acrin_6698,None,ACRIN-6698-928446,1.3.6.1.4.1.14519.5.2.1.7695.4164.221312423812...,1.3.6.1.4.1.14519.5.2.1.7695.4164.113953772496...,10.7937/tcia.kk02-6d95,None,F,2002-12-04,ACRIN-6698_ISPY2_MRI_T2,...,Achieva,2002-12-04,ISPY2: T2W SPAIR DC,301,40,CC BY 4.0,idc-open-data,a0b19173-4974-4c16-8404-488ecd6e8b0e,s3://idc-open-data/a0b19173-4974-4c16-8404-488...,21.32
994029,acrin_6698,None,ACRIN-6698-689480,1.3.6.1.4.1.14519.5.2.1.7695.4164.336722647413...,1.3.6.1.4.1.14519.5.2.1.7695.4164.115308583721...,10.7937/tcia.kk02-6d95,None,F,2002-07-17,ACRIN-6698_ISPY2_MRI_T2,...,Achieva,2002-07-17,ISPY2: T2W SPAIR DC,701,47,CC BY 4.0,idc-open-data,cda68468-410e-4bf7-8c36-69bbd92ac4ca,s3://idc-open-data/cda68468-410e-4bf7-8c36-69b...,25.05
994042,acrin_6698,None,ACRIN-6698-520471,1.3.6.1.4.1.14519.5.2.1.7695.4164.263053910683...,1.3.6.1.4.1.14519.5.2.1.7695.4164.329102493025...,10.7937/tcia.kk02-6d95,None,F,2001-06-15,ACRIN-6698_ISPY2_MRI_T2,...,Avanto,2001-06-15,ISPY2: VOLSER: bi-lateral: PE2,61002,160,CC BY 4.0,idc-open-data,6a26ddb9-30ff-4e63-b045-0fbf6af6827b,s3://idc-open-data/6a26ddb9-30ff-4e63-b045-0fb...,48.25
994043,acrin_6698,None,ACRIN-6698-973837,1.3.6.1.4.1.14519.5.2.1.7695.4164.173598473650...,1.3.6.1.4.1.14519.5.2.1.7695.4164.104285395206...,10.7937/tcia.kk02-6d95,None,F,2001-07-06,ACRIN-6698_ISPY2_MRI_T2,...,Achieva,2001-07-06,ISPY2: T1 FS DYN SENSE 5,701,1225,CC BY 4.0,idc-open-data,d667288a-9282-46ab-b15c-8e625e235c78,s3://idc-open-data/d667288a-9282-46ab-b15c-8e6...,779.03


Now let's take a look at what datasets are contained in your list of series.

In [14]:
summary_breast_mr_t2_df = breast_mr_t2_df.groupby('collection_id').agg(
    Unique_Source_DOIs=('source_DOI', lambda x: x.dropna().unique().tolist()),
    Unique_Licenses=('license_short_name', lambda x: x.dropna().unique().tolist()),
    Unique_Body_Parts=('BodyPartExamined', lambda x: x.dropna().unique().tolist()),
    Unique_Modalities=('Modality', lambda x: x.dropna().unique().tolist()),
    Num_Patients=('PatientID', 'nunique'),
    Num_Studies=('StudyInstanceUID', 'nunique'),
    Num_Series=('SeriesInstanceUID', 'nunique'),
    Total_Series_Size_MB=('series_size_MB', 'sum')
).reset_index()

print("Aggregated Collection Details:")
display(summary_breast_mr_t2_df)

Aggregated Collection Details:


,collection_id,Unique_Source_DOIs,Unique_Licenses,Unique_Body_Parts,Unique_Modalities,Num_Patients,Num_Studies,Num_Series,Total_Series_Size_MB
0,acrin_6698,[10.7937/tcia.kk02-6d95],[CC BY 4.0],[BREAST],[MR],56,157,567,28841.02
1,acrin_contralateral_breast_mr,[10.7937/q1ee-j082],[CC BY 4.0],[BREAST],[MR],720,790,1124,8093.69
2,advanced_mri_breast_lesions,[10.7937/c7x1-yn57],[CC BY 4.0],[BREAST],[MR],632,632,2495,99439.18
3,breast_diagnosis,[10.7937/k9/tcia.2015.sdnrqxxr],[CC BY 3.0],[BREAST],[MR],84,112,121,6729.74
4,breast_mri_nact_pilot,[10.7937/k9/tcia.2016.qhsyhjky],[CC BY 3.0],[BREAST],[MR],36,112,118,613.66
5,cmb_brca,[10.7937/dx22-8j71],[CC BY 4.0],[BREAST],[MR],3,3,4,173.30
6,ea1141,[10.7937/2bas-hr33],[CC BY 4.0],[BREAST],[MR],202,338,396,12859.96
7,ispy1,[10.7937/k9/tcia.2016.hdhpgjlk],[CC BY 3.0],[BREAST],[MR],214,783,1164,8657.30
8,ispy2,[10.7937/tcia.d8z0-9t85],[CC BY 4.0],[BREAST],[MR],241,615,2642,157340.77
9,tcga_brca,[10.7937/k9/tcia.2016.ab2nazrp],[CC BY 3.0],[BREAST],[MR],82,84,145,1384.16


## 4.6 Learn more about your cohort on TCIA

Remember, if you want to go learn more about any of these datasets and find supporting non-DICOM files that go with them you can visit their TCIA landing page.  Just add **https://doi.org/** before the DOI.  E.g. Adding 10.7937/tcia.kk02-6d95 from our results above to form https://doi.org/10.7937/tcia.kk02-6d95 will take you to the ACRIN 6698 landing page.



## 4.7 Adhering to TCIA's Data Usage Policy

Public DICOM datasets from TCIA are generally available under a Creative Commons Attribution license.  Attribution is If you're interested in programmatically generating dataset citations for a particular DOI you can do so using [Crosscite's API](https://citation.doi.org/api-docs.html) as shown below.

In [15]:
import requests

def get_apa_citation(doi):
    url = f"https://citation.crosscite.org/format?doi={doi}&style=apa"
    headers = {"Accept": "text/plain"}  # Ensure plain text response
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        return response.text.strip()
    else:
        return "Error: DOI not found or not available in APA format."

In [16]:
doi = "10.7937/tcia.kk02-6d95"

citation = get_apa_citation(doi)
print(f'The citation for https://doi.org/{doi} is:')
print(citation)

The citation for https://doi.org/10.7937/tcia.kk02-6d95 is:
Newitt, D. C., Partridge, S. C., Zhang, Z., Gibbs, J., Chenevert, T., Rosen, M., Bolan, P., Marques, H., Romanoff, J., Cimino, L., Joe, B. N., Umphrey, H., Ojeda-Fournier, H., Dogan, B., Oh, K. Y., Abe, H., Drukteinis, J., Esserman, L. J., & Hylton, N. M. (2021). ACRIN 6698/I-SPY2 Breast DWI (Version 1) [Data set]. The Cancer Imaging Archive. https://doi.org/10.7937/TCIA.KK02-6D95


# 5 Downloading data
Now let's go over some examples of how to download things after finding some datasets of interest.  Here is a brief summary of how downloads work in idc-index.

In [17]:
help(idc_client.download_from_selection)

Help on method download_from_selection in module idc_index.index:

download_from_selection(downloadDir, dry_run=False, collection_id=None, patientId=None, studyInstanceUID=None, seriesInstanceUID=None, sopInstanceUID=None, crdc_series_uuid=None, quiet=True, show_progress_bar=True, use_s5cmd_sync=False, dirTemplate='%collection_id/%PatientID/%StudyInstanceUID/%Modality_%SeriesInstanceUID', source_bucket_location='aws') method of idc_index.index.IDCClient instance
    Download the files corresponding to the selection. The filtering will be applied in sequence (but does it matter?) by first selecting the collection(s), followed by
    patient(s), study(studies) and series. If no filtering is applied, all the files will be downloaded.

    Args:
        downloadDir: string containing the path to the directory to download the files to
        dry_run: calculates the size of the cohort but download does not start
        collection_id: string or list of strings containing the values of colle

## 5.1 Downloading full collections

In [18]:
# download full collection 'prostate_3t'
idc_client.download_from_selection(collection_id='prostate_3t', downloadDir='.')

## 5.2 Downloading custom search results by UID
Let's take the breast MR T2 cohort we built earlier and download the first 10 scans from it.  

Note that folders are created inside of our specified download directory per the default naming settings `dirTemplate='%collection_id/%PatientID/%StudyInstanceUID/%Modality_%SeriesInstanceUID'`.

In [19]:
idc_client.download_from_selection(
    seriesInstanceUID = list(breast_mr_t2_df['SeriesInstanceUID'].values[:10]),
    downloadDir="./breast_mr_t2")


# 6. Visualizing data before downloading it
You can also leverage IDC's viewer tools to look at specific scans before downloading.  Let's take a look at a few of our breast MR T2 scans.  We'll choose one at random from our previously created `breast_mr_t2_df`.

In [20]:
import random

random_series = random.choice(breast_mr_t2_df['SeriesInstanceUID'].values)
viewer_url = idc_client.get_viewer_URL(seriesInstanceUID=random_series, viewer_selector="ohif_v3")

print(viewer_url)

https://viewer.imaging.datacommons.cancer.gov/v3/viewer/?StudyInstanceUIDs=1.3.6.1.4.1.14519.5.2.1.121063201090068431964235563948431410806&initialSeriesInstanceUID=1.3.6.1.4.1.14519.5.2.1.216269060579884840797222487116078542712


In [21]:
from IPython.display import IFrame
IFrame(viewer_url, width=1024, height=768)

# Acknowledgements
TCIA is funded by the [Cancer Imaging Program (CIP)](https://imaging.cancer.gov/), a part of the United States [National Cancer Institute (NCI)](https://www.cancer.gov/).  It is managed by the [Frederick National Laboratory for Cancer Research (FNLCR)](https://frederick.cancer.gov/) and hosted by the [University of Arkansas for Medical Sciences (UAMS)](https://www.uams.edu/)

If you leverage this notebook or any TCIA datasets in your work, please be sure to comply with the [TCIA Data Usage Policy](https://wiki.cancerimagingarchive.net/x/c4hF). In particular, make sure to cite the DOI(s) for the specific TCIA datasets you used in addition to the following paper!

## TCIA Citation

> Clark, K., Vendt, B., Smith, K., Freymann, J., Kirby, J., Koppel, P., Moore, S., Phillips, S., Maffitt, D., Pringle, M., Tarbox, L., & Prior, F. (2013). The Cancer Imaging Archive (TCIA): Maintaining and Operating a Public Information Repository. Journal of Digital Imaging, 26(6), 1045–1057. https://doi.org/10.1007/s10278-013-9622-7


## IDC Citation

Imaging Data Commons has been funded in whole or in part with Federal funds from the National Cancer Institute, National Institutes of Health, under Task Order No. HHSN26110071 under Contract No. HHSN261201500003l.

If you use IDC in your research, please cite the following publication:

> Fedorov, A., Longabaugh, W. J. R., Pot, D., Clunie, D. A., Pieper, S. D., Gibbs, D. L., Bridge, C., Herrmann, M. D., Homeyer, A., Lewis, R., Aerts, H. J. W., Krishnaswamy, D., Thiriveedhi, V. K., Ciausu, C., Schacherer, D. P., Bontempi, D., Pihl, T., Wagner, U., Farahani, K., Kim, E. & Kikinis, R. National Cancer Institute Imaging Data Commons: Toward Transparency, Reproducibility, and Scalability in Imaging Artificial Intelligence. RadioGraphics (2023). https://doi.org/10.1148/rg.230180